# Hackathon Project: Travel Experience Prediction

**Context:** This notebook aims to predict the `Overall_Experience` of customers traveling. The dataset contains travel and survey data. The goal is to build an accurate machine learning model to classify whether a customer had a positive or negative experience based on various features such as age, travel class, and survey responses.

**Explanation:** Importing required libraries for data manipulation, visualization, and modeling.

In [ ]:
import warnings

warnings.filterwarnings("ignore")
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter("ignore", ConvergenceWarning)

# Libraries to help with reading and manipulating data

import pandas as pd
import numpy as np

# Library to split data
from sklearn.model_selection import train_test_split

# libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 200)
# setting the precision of floating numbers to 5 decimal points
pd.set_option("display.float_format", lambda x: "%.5f" % x)

# To build model for prediction
import statsmodels.stats.api as sms
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from statsmodels.tools.tools import add_constant
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier

# To tune different models
from sklearn.model_selection import GridSearchCV


# To get diferent metric scores
import sklearn.metrics as metrics
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    make_scorer,
)

**Explanation:** Installing necessary packages for our environment.

In [ ]:
!pip install catboost

**Explanation:** Loading the dataset into a pandas DataFrame for analysis.

In [ ]:
travel = pd.read_csv('Traveldata_train_(1)_(1).csv')
survey = pd.read_csv('Surveydata_train_(1)_(1).csv')
df = pd.merge(travel, survey, on='ID')

Traveltest = pd.read_csv('Traveldata_test_(1)_(1).csv')
surveytest = pd.read_csv('Surveydata_test_(1)_(1).csv')
dftest = pd.merge(Traveltest, surveytest, on='ID')

**Explanation:** Exploring the dataset to understand its structure, shape, and basic statistical properties.

In [ ]:
df.dtypes

,0
ID,int64
Gender,object
Customer_Type,object
Age,float64
Type_Travel,object
Travel_Class,object
Travel_Distance,int64
Departure_Delay_in_Mins,float64
Arrival_Delay_in_Mins,float64
Overall_Experience,int64


**Explanation:** Exploring the dataset to understand its structure, shape, and basic statistical properties.

In [ ]:
df.shape

(94379, 25)

**Explanation:** Handling missing values by imputing unknown categories for categorical columns and median values for numerical ones.

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
num_cols = df.select_dtypes(exclude='object').columns

df[cat_cols] = df[cat_cols].fillna('Unknown')
dftest[cat_cols] = dftest[cat_cols].fillna('Unknown')

for col in num_cols:
    if col != 'Overall_Experience':
        median = df[col].median()
        df[col].fillna(median, inplace=True)
        dftest[col].fillna(median, inplace=True)

df.drop('ID', axis=1, inplace=True)


**Explanation:** Exploring the dataset to understand its structure, shape, and basic statistical properties.

In [ ]:
df.describe()

,Age,Travel_Distance,Departure_Delay_in_Mins,Arrival_Delay_in_Mins,Overall_Experience
count,94379.00000,94379.00000,94379.00000,94379.00000,94379.00000
mean,39.41985,1978.88818,14.63825,14.94846,0.54666
std,15.11399,1027.96102,38.12896,38.37769,0.49782
min,7.00000,50.00000,0.00000,0.00000,0.00000
25%,27.00000,1359.00000,0.00000,0.00000,0.00000
50%,40.00000,1923.00000,0.00000,0.00000,1.00000
75%,51.00000,2538.00000,12.00000,13.00000,1.00000
max,85.00000,6951.00000,1592.00000,1584.00000,1.00000


**Explanation:** Binning continuous variables (like Age) into discrete categories to help the model learn non-linear patterns.

In [ ]:
df['Age_bin'] = pd.qcut(
    df['Age'],
    q=5,
    labels=False,
    duplicates='drop'
)

dftest['Age_bin'] = pd.qcut(
    dftest['Age'],
    q=5,
    labels=False,
    duplicates='drop'
)

**Explanation:** Binning continuous variables (like Age) into discrete categories to help the model learn non-linear patterns.

In [ ]:
# calcular bins com base no treino
bins = pd.qcut(df['Age'], q=5, retbins=True, duplicates='drop')[1]

# aplicar no treino
df['Age_bin'] = pd.cut(df['Age'], bins=bins, labels=False, include_lowest=True)

# aplicar no teste (USANDO MESMOS BINS!)
dftest['Age_bin'] = pd.cut(dftest['Age'], bins=bins, labels=False, include_lowest=True)

**Explanation:** Ensuring categorical columns are cast as strings, which is required by certain models like CatBoost.

In [ ]:
df['Age_bin'] = df['Age_bin'].astype(str)
dftest['Age_bin'] = dftest['Age_bin'].astype(str)

**Explanation:** Handling missing values by imputing unknown categories for categorical columns and median values for numerical ones.

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

**Explanation:** Separating the dataset into features (X) and the target variable (y).

In [ ]:
X = df.drop('Overall_Experience', axis=1)
y = df['Overall_Experience']

**Explanation:** Splitting the data into training and validation sets to evaluate the model's generalization.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Explanation:** Getting the numerical indices of categorical columns for CatBoost.

In [ ]:
cat_features = [X.columns.get_loc(col) for col in cat_cols]

**Explanation:** Initializing and training the CatBoost classifier, utilizing early stopping to prevent overfitting.

In [ ]:

from catboost import CatBoostClassifier

model_cat = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100
)

model_cat.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100
)

0:	learn: 0.8393176	test: 0.8429752	best: 0.8429752 (0)	total: 305ms	remaining: 10m 9s
100:	learn: 0.9336583	test: 0.9339903	best: 0.9339903 (99)	total: 32.5s	remaining: 10m 11s
200:	learn: 0.9449691	test: 0.9446387	best: 0.9450625 (197)	total: 1m 2s	remaining: 9m 15s
300:	learn: 0.9497371	test: 0.9490358	best: 0.9490358 (298)	total: 1m 32s	remaining: 8m 39s
400:	learn: 0.9526774	test: 0.9516847	best: 0.9517906 (396)	total: 2m	remaining: 8m 1s
500:	learn: 0.9549025	test: 0.9531151	best: 0.9531151 (499)	total: 2m 31s	remaining: 7m 32s
600:	learn: 0.9565978	test: 0.9547044	best: 0.9548103 (576)	total: 3m 1s	remaining: 7m 1s
700:	learn: 0.9579222	test: 0.9557639	best: 0.9559229 (697)	total: 3m 31s	remaining: 6m 31s
800:	learn: 0.9586639	test: 0.9561878	best: 0.9562937 (789)	total: 4m 2s	remaining: 6m 3s
900:	learn: 0.9595115	test: 0.9561348	best: 0.9563467 (812)	total: 4m 32s	remaining: 5m 32s
1000:	learn: 0.9601870	test: 0.9565056	best: 0.9566645 (963)	total: 5m 2s	remaining: 5m 2s
1100:

CatBoostClassifier(depth=6, eval_metric='Accuracy', iterations=2000, l2_leaf_reg=5, learning_rate=0.03, random_seed=42, verbose=100)

**Explanation:** Evaluating the model's accuracy on both the training and validation sets to check for overfitting.

In [ ]:
from sklearn.metrics import accuracy_score

# Train
train_pred = model_cat.predict(X_train)
train_acc = accuracy_score(y_train, train_pred)

# Validation
val_pred = model_cat.predict(X_val)
val_acc = accuracy_score(y_val, val_pred)

print("Train Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Gap:", train_acc - val_acc)

Train Accuracy: 0.9613922625591036
Validation Accuracy: 0.9573002754820936
Gap: 0.004091987077009973


**Explanation:** Executing data processing and modeling steps.

In [ ]:
proba = model_cat.predict_proba(X_val)[:,1]

**Explanation:** Executing data processing and modeling steps.

In [ ]:
from sklearn.metrics import accuracy_score

best_acc = 0
best_t = 0

for t in np.arange(0.40, 0.60, 0.01):
    pred = (proba > t).astype(int)
    acc = accuracy_score(y_val, pred)

    print(f"T={t:.2f} | Acc={acc:.5f}")

    if acc > best_acc:
        best_acc = acc
        best_t = t

print("\nBest Threshold:", best_t)
print("Best Accuracy:", best_acc)

T=0.40 | Acc=0.95364
T=0.41 | Acc=0.95391
T=0.42 | Acc=0.95433
T=0.43 | Acc=0.95518
T=0.44 | Acc=0.95497
T=0.45 | Acc=0.95566
T=0.46 | Acc=0.95613
T=0.47 | Acc=0.95656
T=0.48 | Acc=0.95693
T=0.49 | Acc=0.95704
T=0.50 | Acc=0.95730
T=0.51 | Acc=0.95719
T=0.52 | Acc=0.95688
T=0.53 | Acc=0.95704
T=0.54 | Acc=0.95688
T=0.55 | Acc=0.95666
T=0.56 | Acc=0.95682
T=0.57 | Acc=0.95698
T=0.58 | Acc=0.95682
T=0.59 | Acc=0.95709

Best Threshold: 0.5000000000000001
Best Accuracy: 0.9573002754820936


##Preparar teste

**Explanation:** Filtering the test dataset to include only the columns that were used during training.

In [ ]:
X_test_final = dftest[X.columns].copy()

**Explanation:** Handling missing values by imputing unknown categories for categorical columns and median values for numerical ones.

In [ ]:
for col in cat_cols:
    X_test_final[col] = X_test_final[col].fillna('Unknown').astype(str)

**Explanation:** Handling missing values by imputing unknown categories for categorical columns and median values for numerical ones.

In [ ]:
num_cols = X.select_dtypes(exclude='object').columns

for col in num_cols:
    X_test_final[col] = X_test_final[col].fillna(X[col].median())

**Explanation:** Setting the best probability threshold found during our search.

In [ ]:
proba_test = model_cat.predict_proba(X_test_final)[:,1]

BEST_THRESHOLD = 0.50  # ou o melhor que você achou

preds_split = (proba_test > BEST_THRESHOLD).astype(int)

**Explanation:** Initializing and training the CatBoost classifier, utilizing early stopping to prevent overfitting.

In [ ]:
model_cat_full = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100
)

model_cat_full.fit(
    X, y,
    cat_features=cat_features
)

0:	learn: 0.8336176	total: 871ms	remaining: 29m 1s
100:	learn: 0.9348584	total: 34.1s	remaining: 10m 41s
200:	learn: 0.9444792	total: 1m 8s	remaining: 10m 14s
300:	learn: 0.9502644	total: 1m 44s	remaining: 9m 49s
400:	learn: 0.9536338	total: 2m 18s	remaining: 9m 10s
500:	learn: 0.9559860	total: 2m 52s	remaining: 8m 35s
600:	learn: 0.9573210	total: 3m 26s	remaining: 8m 1s
700:	learn: 0.9584018	total: 4m	remaining: 7m 26s
800:	learn: 0.9592600	total: 4m 34s	remaining: 6m 51s
900:	learn: 0.9599699	total: 5m 8s	remaining: 6m 16s
1000:	learn: 0.9606798	total: 5m 43s	remaining: 5m 42s
1100:	learn: 0.9611884	total: 6m 17s	remaining: 5m 8s
1200:	learn: 0.9618241	total: 6m 52s	remaining: 4m 34s
1300:	learn: 0.9621526	total: 7m 27s	remaining: 4m
1400:	learn: 0.9627883	total: 8m 2s	remaining: 3m 26s
1500:	learn: 0.9631804	total: 8m 39s	remaining: 2m 52s
1600:	learn: 0.9637313	total: 9m 15s	remaining: 2m 18s
1700:	learn: 0.9641552	total: 9m 50s	remaining: 1m 43s
1800:	learn: 0.9645366	total: 10m 2

CatBoostClassifier(depth=6, eval_metric='Accuracy', iterations=2000, l2_leaf_reg=5, learning_rate=0.03, random_seed=42, verbose=100)

**Explanation:** Applying the optimal threshold to generate final class predictions.

In [ ]:
proba_test_full = model_cat_full.predict_proba(X_test_final)[:,1]

preds_full = (proba_test_full > BEST_THRESHOLD).astype(int)

**Explanation:** Applying the optimal threshold to generate final class predictions.

In [ ]:
proba_test_full = model_cat_full.predict_proba(X_test_final)[:,1]

preds_full = (proba_test_full > BEST_THRESHOLD).astype(int)

**Explanation:** Loading the dataset into a pandas DataFrame for analysis.

In [ ]:
sub2 = pd.read_csv('Sample_Submission_(1)_(1).csv')
sub2['Overall_Experience'] = preds_full
sub2.to_csv('submission_full.csv', index=False)

**Explanation:** Loading the dataset into a pandas DataFrame for analysis.

In [ ]:
sub1 = pd.read_csv('Sample_Submission_(1)_(1).csv')
sub1['Overall_Experience'] = preds_split
sub1.to_csv('submission_split.csv', index=False)

**Explanation:** Loading the dataset into a pandas DataFrame for analysis.

In [ ]:
test112 = pd.read_csv('submission_full.csv')
test11 = pd.read_csv('submission_split.csv')

**Explanation:** Previewing the generated data structure.

In [ ]:
test112.head()

,ID,Overall_Experience
0,99900001,1
1,99900002,1
2,99900003,1
3,99900004,0
4,99900005,1


**Explanation:** Previewing the generated data structure.

In [ ]:
test11.head()

,ID,Overall_Experience
0,99900001,1
1,99900002,1
2,99900003,1
3,99900004,0
4,99900005,1
